In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
import pandas as pd
from datetime import datetime
import fnmatch


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion_cor.npz')
xd, yd = s['xd'], s['yd']
xu, yu = s['xu'], s['yu']

df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers_cor.csv', skipinitialspace=True).dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/wcs.csv', skipinitialspace=True).dropna()

dates = np.array([datetime.fromisoformat(date) for date in df['date']])
car_rot = df['car_rot'].to_numpy()
hgln = df['hgln_obs'].to_numpy()



In [3]:
def calc_fluxes_(file, dlat=1, mu_thr=0.1):
    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    did = int(file.split('_')[-1].split('.')[0])

    view = View.from_header(header)
    view.update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=view.crota + 0.25, rsun=ru_sun[dids == did][0], inplace=True)

    transform = view.to_carrington(correct_mu=True, mu_thr=mu_thr) + ToSpherical()
    grid, mu = transform(crop_grid(xu, yu, header))

    Br = data / mu
    lat_map = grid[0] // dlat * dlat

    lat = np.arange(-90,90,dlat)
    weight = np.zeros_like(lat).astype(np.float32)
    flux_density = np.zeros_like(lat).astype(np.float32)

    for i, lat_ in enumerate(lat):
        t = (lat_map == lat_)
        nt = np.sum(t)

        if nt > 0:
            Br_ = Br[t]
            W_ = mu[t] ** 4 / mu[t]

            weight[i] = np.nansum(W_)
            flux_density[i] = np.nansum(W_ * Br_) / weight[i]

    return lat, weight, flux_density


def calc_fluxes(files, **kwargs):
    x, mean, w_sum, w_sum2, S = 0., 0., 0. ,0., 0.

    for file in files:
        print(file)

        x, w, y = calc_fluxes_(file, **kwargs)

        w_sum += w
        w_sum2 += w ** 2
        mean_old = mean + 0.
        mean += np.nan_to_num((w / w_sum) * (y - mean))
        S += np.nan_to_num(w * (y - mean_old) * (y - mean))

    S *= w_sum2 / w_sum ** 3
    return x, mean, S


In [21]:
files_ = sorted(glob.glob('/home/ulyanov/data/solo/phi/2024_2025/*'))

files = []
for file in files_:
    date = int(file.split('/')[-1].split('_')[3][:8])
    #if date > 20250425 and date < 20250527:
    #if date > 20250818 and date < 20251000:
    if date > 20241026 and date < 20241127:
        files.append(file)

files

['/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T031503_V202602220112_0450270501.fits.gz',
 '/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T061503_V202602220112_0450270502.fits.gz',
 '/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T091503_V202602220112_0450270503.fits.gz',
 '/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T121503_V202602220112_0450270504.fits.gz',
 '/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T151503_V202602220112_0450270505.fits.gz',
 '/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T181503_V202602220112_0450270506.fits.gz',
 '/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T211503_V202602220112_0450270507.fits.gz',
 '/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241030T001503_V202602220112_0450300501.fits.gz',
 '/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241030T031503_V202602220112_0450300502.fi

In [22]:
with fits.open(files[0]) as hdul:
    header = hdul[0].header.copy()

header['CRLN_OBS']

211.80923

In [23]:
with fits.open(files[-1]) as hdul:
    header = hdul[0].header.copy()

header['CRLN_OBS']

174.33311

In [24]:
lat, flux_density, variance = calc_fluxes(files)

/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T031503_V202602220112_0450270501.fits.gz


/tmp/ipykernel_142532/1022253968.py:30: RuntimeWarning: invalid value encountered in scalar divide
  flux_density[i] = np.nansum(W_ * Br_) / weight[i]
/tmp/ipykernel_142532/1022253968.py:46: RuntimeWarning: invalid value encountered in divide
  mean += np.nan_to_num((w / w_sum) * (y - mean))


/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T061503_V202602220112_0450270502.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T091503_V202602220112_0450270503.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T121503_V202602220112_0450270504.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T151503_V202602220112_0450270505.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T181503_V202602220112_0450270506.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241027T211503_V202602220112_0450270507.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241030T001503_V202602220112_0450300501.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241030T031503_V202602220112_0450300502.fits.gz
/home/ulyanov/data/solo/phi/2024_2025/solo_L2_phi-fdt-blos_20241030T061503_V202602220112_0450300503.fits.gz
/home/ulyanov/data/solo/phi/

/tmp/ipykernel_142532/1022253968.py:49: RuntimeWarning: invalid value encountered in divide
  S *= w_sum2 / w_sum ** 3


In [25]:
np.savez('fluxes_202411fdt.npz', lat=lat, flux_density=flux_density, variance=variance)